# Barge-in and Interruptions

Drive the barge-in controller by hand. Predict each outcome before you run the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

cfg = main.BargeInConfig()   # LiveKit defaults: min_duration 0.5, timeout 2.0, resume True
for words, duration in [(('khoan', 'đã'), 0.9), (('không',), 0.3), (('dạ',), 0.3), ((), 0.6)]:
    event = main.OverlapEvent('?', 2.0, duration, words, True)
    out = main.simulate_event(event, 6.0, cfg)
    print(words or '(cough)', '->', out.action, out.trigger, out.silenced_at)

## Compare controllers on the same overlaps

Try `min_duration=0.3`, `min_words=1`, or `false_interruption_timeout=None`.

In [ ]:
events = main.make_events(400, seed=0)
for name, c in [('defaults', main.BargeInConfig()),
                ('min_words=2', main.BargeInConfig(min_words=2)),
                ('adaptive', main.BargeInConfig(mode='adaptive'))]:
    m = main.evaluate(events, c)
    print(f"{name:<12} missed {m['missed_rate']:.0%}  false stops {m['false_stop_rate']:.0%}  "
          f"false pauses {m['false_pause_rate']:.0%}  median stop {m['median_stop_latency']:.2f} s")

## Echo and the guard window

How long a guard window do you need, and what does it cost real interruptions early in a reply?

In [ ]:
echo = main.make_events(400, seed=1, include_echo=True)
for guard in (0.0, 1.0, 2.0, 3.0):
    m = main.evaluate(echo, main.BargeInConfig(mode='adaptive', guard_s=guard))
    print(f"guard {guard:.0f} s: false stops {m['false_stop_rate']:.0%}, missed {m['missed_rate']:.0%}")

## What the caller heard

Move the interruption point and see what stays in the chat history.

In [ ]:
timings = main.word_timings(main.AGENT_REPLY, 6.0)
for t in (1.0, 2.4, 4.0):
    heard, unheard = main.truncate_to_heard(main.AGENT_REPLY, timings, t)
    print(f'{t:.1f} s heard: {heard!r}')